In [1]:
import pandas as pd

In [2]:
df_metadata = pd.read_csv("merge_ecac/metadata.csv")

In [3]:
df_metadata.shape

(1348, 27)

In [4]:
df_metadata.head()

,Audio_Song,Lyric_Song,Quadrant,AllMusic Id,AllMusic Extraction Date,Artist,Title,Relevance,Year,LowestYear,...,ThemeWeights,Styles,StyleWeights,AppearancesTrackIDs,AppearancesAlbumIDs,Sample,SampleURL,ActualYear,num_Genres,num_MoodsAll
0,MT0002007417,MT0002007417,Q3,MT0002007417,Old,Marlene Dietrich,Falling in Love Again,1.040315,27/07/2004,1994-??-??,...,"5,6,7,8,8,9,9","Cabaret,Torch Songs,Traditional Pop","7,8,8","MT0002007417,MT0033924988","MW0000204999,MW0001114125",1.0,http://rovimusic.rovicorp.com/playback.mp3?c=j...,2004,NaN,NaN
1,MT0009263768,MT0009263768,Q3,MT0009263768,Old,George Strait,Four Down and Twelve Across,NaN,NaN,NaN,...,"9,9,9","Contemporary Country,New Traditionalist","5,9",NaN,NaN,1.0,http://rovimusic.rovicorp.com/playback.mp3?c=p...,NaN,NaN,NaN
2,MT0030214520,MT0030214520,Q2,MT0030214520,Old,Gorilla Biscuits,Forgotten,1.020054,1989-??-??,1988-??-??,...,NaN,"Alternative/Indie Rock,Contemporary Pop/Rock,I...","7,7,7,9,9","MT0002293082,MT0003076885,MT0004145917,MT00051...","MW0000199609,MW0000199609,MW0000199609,MW00001...",1.0,http://rovimusic.rovicorp.com/playback.mp3?c=W...,1989,NaN,NaN
3,MT0000097948,MT0000097948,Q4,MT0000097948,Old,Nat King Cole,"Baby, Baby All the Time",1.081924,07/10/2003,NaN,...,"7,8,9","Swing,Traditional Pop,Vocal Jazz","7,8,9","MT0000097948,MT0006106958,MT0006392619,MT00072...","MW0000027198,MW0000027198,MW0000044908,MW00000...",1.0,http://rovimusic.rovicorp.com/playback.mp3?c=K...,2003,NaN,NaN
4,MT0009005900,MT0009005900,Q1,MT0009005900,Old,Merle Haggard,I Made the Prison Band,NaN,1967-??-??,1967-??-??,...,NaN,Traditional Country,5,"MT0000022647,MT0004898729,MT0004988534,MT00057...","MW0000015847,MW0000176873,MW0000176873,MW00004...",1.0,http://rovimusic.rovicorp.com/playback.mp3?c=d...,1967,NaN,NaN


# 4. Limpeza dos Dados

## 4.1 Duplicados

### 4.1.1 Identificação de linhas idênticas

O primeiro passo consiste em identificar linhas que sejam completamente idênticas em todas as colunas.

In [6]:
duplicate_mask = df_metadata.duplicated(keep=False)

identical_rows = df_metadata[duplicate_mask]

print("Total rows:", len(df_metadata))
print("Rows involved in exact duplication:", len(identical_rows))
print("Extra duplicate copies:", df_metadata.duplicated().sum())

Total rows: 1348
Rows involved in exact duplication: 50
Extra duplicate copies: 25


In [7]:
display(
    identical_rows[
        ["Audio_Song", "Lyric_Song", "Artist", "Title", "Quadrant"]
    ].sort_values("Audio_Song")
)

,Audio_Song,Lyric_Song,Artist,Title,Quadrant
971,A071-70,L121-70,Ben Folds Five,Brick,Q3
131,A071-70,L121-70,Ben Folds Five,Brick,Q3
103,A132-121,L022-121,The Ten Tenors,"Bee Gees Medley: Staying Alive, You Should Be ...",Q1
556,A132-121,L022-121,The Ten Tenors,"Bee Gees Medley: Staying Alive, You Should Be ...",Q1
1108,MT0000408441,MT0000408441,Lightnin' Hopkins,Give Me Back That Wig (Wig Wearing Woman),Q3
338,MT0000408441,MT0000408441,Lightnin' Hopkins,Give Me Back That Wig (Wig Wearing Woman),Q3
544,MT0000685538,MT0000685538,Carl Perkins,Let the Jukebox Keep on Playing,Q3
640,MT0000685538,MT0000685538,Carl Perkins,Let the Jukebox Keep on Playing,Q3
179,MT0000886390,MT0000886390,Dolly Parton,Randy,Q1
378,MT0000886390,MT0000886390,Dolly Parton,Randy,Q1


**Resposta:** O conjunto de dados contém 25 cópias exatamente duplicadas. Estas correspondem a 25 pares de linhas completamente idênticas, o que significa que 50 linhas estão envolvidas em duplicação exata. Portanto, 25 linhas são cópias redundantes.

### 4.1.2 Identificação da mesma gravação

Para determinar se diferentes linhas representam a mesma gravação, serão comparados quatro tipos de evidência: as colunas de identificação, o artista e o título, a letra e o áudio.

In [8]:
df_metadata.columns.tolist()

['Audio_Song',
 'Lyric_Song',
 'Quadrant',
 'AllMusic Id',
 'AllMusic Extraction Date',
 'Artist',
 'Title',
 'Relevance',
 'Year',
 'LowestYear',
 'Duration',
 'Moods',
 'MoodsAll',
 'MoodsAllWeights',
 'Genres',
 'GenreWeights',
 'Themes',
 'ThemeWeights',
 'Styles',
 'StyleWeights',
 'AppearancesTrackIDs',
 'AppearancesAlbumIDs',
 'Sample',
 'SampleURL',
 'ActualYear',
 'num_Genres',
 'num_MoodsAll']

In [9]:
identificadores = ["Audio_Song", "Lyric_Song", "AllMusic Id"]

for coluna in identificadores:
    print(f"{coluna}:")
    print(f"  Valores únicos: {df_metadata[coluna].nunique(dropna=True)}")
    print(f"  Valores ausentes: {df_metadata[coluna].isna().sum()}")
    print(f"  Valores repetidos: {df_metadata[coluna].duplicated().sum()}")
    print()

Audio_Song:
  Valores únicos: 1323
  Valores ausentes: 0
  Valores repetidos: 25

Lyric_Song:
  Valores únicos: 1323
  Valores ausentes: 0
  Valores repetidos: 25

AllMusic Id:
  Valores únicos: 1320
  Valores ausentes: 0
  Valores repetidos: 28



In [10]:
allmusic_repetidos = df_metadata[
    df_metadata.duplicated("AllMusic Id", keep=False)
].sort_values("AllMusic Id")

display(
    allmusic_repetidos[
        ["AllMusic Id", "Audio_Song", "Lyric_Song", "Artist", "Title", "Quadrant"]
    ]
)

,AllMusic Id,Audio_Song,Lyric_Song,Artist,Title,Quadrant
1108,MT0000408441,MT0000408441,MT0000408441,Lightnin' Hopkins,Give Me Back That Wig (Wig Wearing Woman),Q3
338,MT0000408441,MT0000408441,MT0000408441,Lightnin' Hopkins,Give Me Back That Wig (Wig Wearing Woman),Q3
544,MT0000685538,MT0000685538,MT0000685538,Carl Perkins,Let the Jukebox Keep on Playing,Q3
640,MT0000685538,MT0000685538,MT0000685538,Carl Perkins,Let the Jukebox Keep on Playing,Q3
378,MT0000886390,MT0000886390,MT0000886390,Dolly Parton,Randy,Q1
179,MT0000886390,MT0000886390,MT0000886390,Dolly Parton,Randy,Q1
353,MT0001189913,MT0001189913,MT0001189913,Kay Kyser,"(I Got Spurs That) Jingle, Jangle, Jingle",Q1
701,MT0001189913,MT0001189913,MT0001189913,Kay Kyser,"(I Got Spurs That) Jingle, Jangle, Jingle",Q1
1283,MT0001841416,MT0001841416,MT0001841416,Lightnin' Hopkins,Shotgun (Blues),Q3
1200,MT0001841416,MT0001841416,MT0001841416,Lightnin' Hopkins,Shotgun (Blues),Q3


In [12]:
# Remove apenas as cópias exatamente idênticas para investigar
# repetições de identificadores entre registros diferentes
df_sem_duplicados_exatos = df_metadata.drop_duplicates()

print("Linhas originais:", len(df_metadata))
print("Linhas após remover cópias exatas:", len(df_sem_duplicados_exatos))

Linhas originais: 1348
Linhas após remover cópias exatas: 1323


In [13]:
allmusic_conflitos = df_sem_duplicados_exatos[
    df_sem_duplicados_exatos.duplicated("AllMusic Id", keep=False)
].sort_values("AllMusic Id")

print("Linhas com AllMusic Id repetido após remover duplicados exatos:",
      len(allmusic_conflitos))

print("AllMusic Id distintos envolvidos:",
      allmusic_conflitos["AllMusic Id"].nunique())

display(
    allmusic_conflitos[
        ["AllMusic Id", "Audio_Song", "Lyric_Song", "Artist", "Title", "Quadrant"]
    ]
)

Linhas com AllMusic Id repetido após remover duplicados exatos: 4
AllMusic Id distintos envolvidos: 1


,AllMusic Id,Audio_Song,Lyric_Song,Artist,Title,Quadrant
97,not_found,A170,L170,Silverchair,Suicidal Dream,Q3
611,not_found,A179,L179,Monica Brown,One People One Land,Q4
935,not_found,A109-138,L159-138,Diana Krall,Love is Where You Are,Q4
1347,not_found,A165,L165,Amata,Water Song,Q4


#### Evidência pelos identificadores

Foram inicialmente analisadas as colunas `Audio_Song`, `Lyric_Song` e `AllMusic Id`. Os identificadores `Audio_Song` e `Lyric_Song` apresentam 25 repetições cada, correspondentes às 25 cópias exatas identificadas anteriormente.

A coluna `AllMusic Id` apresenta 28 repetições. Contudo, após a remoção das cópias exatas, verificou-se que as três repetições adicionais são explicadas por quatro registos distintos contendo o valor `not_found`. Este valor representa a ausência de um identificador válido e, portanto, não constitui evidência de que estas quatro linhas correspondam à mesma gravação.

In [14]:
for coluna in ["Audio_Song", "Lyric_Song"]:
    repetidos = df_sem_duplicados_exatos[
        df_sem_duplicados_exatos.duplicated(coluna, keep=False)
    ]

    print(f"{coluna}:")
    print(f"  Linhas envolvidas em repetição: {len(repetidos)}")
    print(f"  Identificadores repetidos: {repetidos[coluna].nunique()}")
    print()

Audio_Song:
  Linhas envolvidas em repetição: 0
  Identificadores repetidos: 0

Lyric_Song:
  Linhas envolvidas em repetição: 0
  Identificadores repetidos: 0



In [16]:
artist_title_repetidos = df_sem_duplicados_exatos[
    df_sem_duplicados_exatos.duplicated(
        subset=["Artist", "Title"],
        keep=False
    )
].sort_values(["Artist", "Title"])

print("Linhas com a mesma combinação Artist + Title:",
      len(artist_title_repetidos))

print("Combinações Artist + Title repetidas:",
      artist_title_repetidos[["Artist", "Title"]].drop_duplicates().shape[0])

display(
    artist_title_repetidos[
        ["Artist", "Title", "Audio_Song", "Lyric_Song", "AllMusic Id", "Quadrant"]
    ]
)

Linhas com a mesma combinação Artist + Title: 0
Combinações Artist + Title repetidas: 0


,Artist,Title,Audio_Song,Lyric_Song,AllMusic Id,Quadrant


In [17]:
from pathlib import Path

lyrics_path = Path("../../dataset/lyrics")

print("A pasta existe:", lyrics_path.exists())

arquivos_lyrics = list(lyrics_path.rglob("*"))
arquivos_lyrics = [arquivo for arquivo in arquivos_lyrics if arquivo.is_file()]

print("Quantidade de arquivos encontrados:", len(arquivos_lyrics))

for arquivo in arquivos_lyrics[:10]:
    print(arquivo)

A pasta existe: True
Quantidade de arquivos encontrados: 2217
..\..\dataset\lyrics\.DS_Store
..\..\dataset\lyrics\Q1\L004-120.txt
..\..\dataset\lyrics\Q1\L005-84.txt
..\..\dataset\lyrics\Q1\L018-109.txt
..\..\dataset\lyrics\Q1\L022-121.txt
..\..\dataset\lyrics\Q1\L036-118.txt
..\..\dataset\lyrics\Q1\L038-102.txt
..\..\dataset\lyrics\Q1\L042-108.txt
..\..\dataset\lyrics\Q1\L043-130.txt
..\..\dataset\lyrics\Q1\L050-149.txt


In [18]:
print("Primeiros 20 valores de Lyric_Song:")
print(df_metadata["Lyric_Song"].head(20).tolist())

print("\nPrimeiros 20 nomes de arquivos de letras:")
for arquivo in arquivos_lyrics[:20]:
    print(arquivo.name)

Primeiros 20 valores de Lyric_Song:
['MT0002007417', 'MT0009263768', 'MT0030214520', 'MT0000097948', 'MT0009005900', 'MT0031826295', 'MT0000092637', 'MT0007860263', 'MT0010715611', 'MT0011870505', 'MT0010100278', 'MT0004397754', 'MT0001590644', 'MT0000325093', 'MT0015541501', 'MT0000638634', 'MT0001610102', 'MT0001266436', 'MT0004400843', 'MT0006169551']

Primeiros 20 nomes de arquivos de letras:
.DS_Store
L004-120.txt
L005-84.txt
L018-109.txt
L022-121.txt
L036-118.txt
L038-102.txt
L042-108.txt
L043-130.txt
L050-149.txt
L055.txt
L061.txt
L064.txt
L069.txt
L072.txt
L074.txt
L089.txt
L092.txt
L093.txt
L097.txt


In [19]:
# Considerar apenas arquivos .txt de letras
arquivos_txt = [arquivo for arquivo in arquivos_lyrics if arquivo.suffix.lower() == ".txt"]

# Nome do arquivo sem a extensão .txt
nomes_lyrics = {arquivo.stem for arquivo in arquivos_txt}

# Verificar quais Lyric_Song possuem um arquivo correspondente
tem_arquivo = df_metadata["Lyric_Song"].isin(nomes_lyrics)

print("Arquivos .txt encontrados:", len(arquivos_txt))
print("Linhas do metadata com arquivo de letra correspondente:", tem_arquivo.sum())
print("Linhas sem arquivo correspondente:", (~tem_arquivo).sum())

print("\nExemplos com correspondência:")
print(df_metadata.loc[tem_arquivo, "Lyric_Song"].head(10).tolist())

print("\nExemplos sem correspondência:")
print(df_metadata.loc[~tem_arquivo, "Lyric_Song"].head(10).tolist())

Arquivos .txt encontrados: 2216
Linhas do metadata com arquivo de letra correspondente: 1348
Linhas sem arquivo correspondente: 0

Exemplos com correspondência:
['MT0002007417', 'MT0009263768', 'MT0030214520', 'MT0000097948', 'MT0009005900', 'MT0031826295', 'MT0000092637', 'MT0007860263', 'MT0010715611', 'MT0011870505']

Exemplos sem correspondência:
[]


In [20]:
from collections import Counter

contagem_nomes = Counter(arquivo.stem for arquivo in arquivos_txt)

nomes_repetidos = {
    nome: quantidade
    for nome, quantidade in contagem_nomes.items()
    if quantidade > 1
}

print("Nomes de arquivos .txt únicos:", len(contagem_nomes))
print("Nomes que aparecem em mais de uma pasta:", len(nomes_repetidos))

list(nomes_repetidos.items())[:20]

Nomes de arquivos .txt únicos: 2216
Nomes que aparecem em mais de uma pasta: 0


[]

In [21]:
# Criar um mapa: identificador da letra -> caminho do arquivo
mapa_lyrics = {
    arquivo.stem: arquivo
    for arquivo in arquivos_txt
}

print("Quantidade de letras no mapa:", len(mapa_lyrics))

# Testar com a primeira música do metadata
primeiro_id = df_metadata.loc[0, "Lyric_Song"]

print("Lyric_Song:", primeiro_id)
print("Arquivo:", mapa_lyrics[primeiro_id])

Quantidade de letras no mapa: 2216
Lyric_Song: MT0002007417
Arquivo: ..\..\dataset\lyrics\Q3\MT0002007417.txt


In [22]:
arquivo_teste = mapa_lyrics[primeiro_id]

with open(arquivo_teste, "r", encoding="utf-8", errors="replace") as f:
    letra_teste = f.read()

print(letra_teste[:1000])

Falling in love again
Never wanted to
What am I to do?
Can't help it

Love's always been my game
Play it how I may
I was made that way
Can't help it

Men cluster to me like moths around a flame
And if their wings burn, I know I'm not to blame

Falling in love again
Never wanted to
What am I to do?
Can't help it

Fallin' in love again
Never wanted to
What am I to do?
Can't help it

Love's always been my game
Play it how I may
I was made that way
Can't help it

Men cluster to me like moths around a flame
And if their wings burn, I know I'm not to blame

Fallin' in love again
Never wanted to
What am I to do?
Can't help it

Men cluster to me like moths around a flame
And if their wings burn, I know I'm not to blame

Fallin' in love again
Never wanted to
What am I to do?
Can't help it


In [23]:
import hashlib
import re

def normalizar_letra(texto):
    # Coloca tudo em minúsculas
    texto = texto.lower()

    # Remove diferenças de espaços, quebras de linha e tabs
    texto = re.sub(r"\s+", " ", texto)

    # Remove espaços no início e no final
    return texto.strip()


def hash_letra(lyric_id):
    caminho = mapa_lyrics[lyric_id]

    with open(caminho, "r", encoding="utf-8", errors="replace") as f:
        texto = f.read()

    texto_normalizado = normalizar_letra(texto)

    return hashlib.sha256(
        texto_normalizado.encode("utf-8")
    ).hexdigest()

In [24]:
df_letras = df_sem_duplicados_exatos.copy()

df_letras["Lyric_Hash"] = df_letras["Lyric_Song"].apply(hash_letra)

print("Registros analisados:", len(df_letras))
print("Letras únicas pelo conteúdo:", df_letras["Lyric_Hash"].nunique())

Registros analisados: 1323
Letras únicas pelo conteúdo: 1314


In [25]:
letras_repetidas = df_letras[
    df_letras.duplicated("Lyric_Hash", keep=False)
].sort_values("Lyric_Hash")

print("Linhas envolvidas em letras de conteúdo idêntico:",
      len(letras_repetidas))

print("Conteúdos de letra repetidos:",
      letras_repetidas["Lyric_Hash"].nunique())

display(
    letras_repetidas[
        ["Artist", "Title", "Audio_Song", "Lyric_Song", "Quadrant"]
    ]
)

Linhas envolvidas em letras de conteúdo idêntico: 18
Conteúdos de letra repetidos: 9


,Artist,Title,Audio_Song,Lyric_Song,Quadrant
155,Hank Williams,I'm So Lonesome,MT0001134958,MT0001134958,Q3
998,Hank Williams,"I'm So Lonesome, I Could Cry",MT0015665796,MT0015665796,Q3
83,The Isley Brothers,Twist and Shout,MT0002672499,MT0002672499,Q1
679,The Beatles,Twist and Shout,A146-118,L036-118,Q1
67,Prince,U Got the Look,MT0004444570,MT0004444570,Q1
1160,Prince,U GOT THE LOOK,A005,L055,Q1
455,Sparks,Amateur Hour,MT0011847514,MT0011847514,Q1
1199,Erasure,Amateur Hour,MT0007067293,MT0007067293,Q1
165,Godsmack,I F****** Hate You,MT0003503243,MT0003503243,Q2
1058,Godsmack,I Fucking Hate You,A028,L078,Q2


In [26]:
print("Registros analisados:", len(df_letras))
print("Letras únicas pelo conteúdo:", df_letras["Lyric_Hash"].nunique())
print("Linhas envolvidas em letras idênticas:", len(letras_repetidas))
print("Grupos de letras idênticas:", letras_repetidas["Lyric_Hash"].nunique())

Registros analisados: 1323
Letras únicas pelo conteúdo: 1314
Linhas envolvidas em letras idênticas: 18
Grupos de letras idênticas: 9


#### Comparação das letras

Após a remoção dos duplicados exatos, foram analisados 1.323 registros. A comparação do conteúdo normalizado dos arquivos de letras identificou 1.314 letras distintas. Foram encontrados 18 registros, organizados em 9 grupos, que possuem conteúdo de letra idêntico.

No entanto, uma letra idêntica não é suficiente para concluir que dois registros representam a mesma gravação. Foram encontrados casos em que artistas diferentes apresentam letras idênticas, como "Twist and Shout", interpretada por The Isley Brothers e The Beatles. Portanto, a igualdade das letras pode indicar a mesma música ou composição, mas não necessariamente a mesma gravação.

In [27]:
audio_path = Path("../../dataset/audio")

print("A pasta de áudio existe:", audio_path.exists())

arquivos_audio = [
    arquivo
    for arquivo in audio_path.rglob("*")
    if arquivo.is_file()
]

print("Quantidade de arquivos encontrados:", len(arquivos_audio))

for arquivo in arquivos_audio[:20]:
    print(arquivo)

A pasta de áudio existe: True
Quantidade de arquivos encontrados: 2217
..\..\dataset\audio\.DS_Store
..\..\dataset\audio\Q1\A005.mp3
..\..\dataset\audio\Q1\A011.mp3
..\..\dataset\audio\Q1\A014.mp3
..\..\dataset\audio\Q1\A019.mp3
..\..\dataset\audio\Q1\A022.mp3
..\..\dataset\audio\Q1\A024.mp3
..\..\dataset\audio\Q1\A039.mp3
..\..\dataset\audio\Q1\A042.mp3
..\..\dataset\audio\Q1\A043.mp3
..\..\dataset\audio\Q1\A047.mp3
..\..\dataset\audio\Q1\A050.mp3
..\..\dataset\audio\Q1\A064-116.mp3
..\..\dataset\audio\Q1\A077-122.mp3
..\..\dataset\audio\Q1\A086-123.mp3
..\..\dataset\audio\Q1\A092-96.mp3
..\..\dataset\audio\Q1\A094-110.mp3
..\..\dataset\audio\Q1\A095-113.mp3
..\..\dataset\audio\Q1\A100-124.mp3
..\..\dataset\audio\Q1\A101-111.mp3


In [28]:
# Manter apenas os arquivos MP3
arquivos_mp3 = [
    arquivo
    for arquivo in arquivos_audio
    if arquivo.suffix.lower() == ".mp3"
]

# Criar conjunto com os nomes dos arquivos sem a extensão
nomes_audio = {arquivo.stem for arquivo in arquivos_mp3}

# Verificar correspondência com Audio_Song
tem_audio = df_metadata["Audio_Song"].isin(nomes_audio)

print("Arquivos .mp3 encontrados:", len(arquivos_mp3))
print("Nomes de arquivos .mp3 únicos:", len(nomes_audio))
print("Linhas do metadata com áudio correspondente:", tem_audio.sum())
print("Linhas sem áudio correspondente:", (~tem_audio).sum())

print("\nExemplos sem correspondência:")
print(
    df_metadata.loc[~tem_audio, "Audio_Song"]
    .head(20)
    .tolist()
)

Arquivos .mp3 encontrados: 2216
Nomes de arquivos .mp3 únicos: 2216
Linhas do metadata com áudio correspondente: 1348
Linhas sem áudio correspondente: 0

Exemplos sem correspondência:
[]


In [29]:
# Criar mapa: Audio_Song -> caminho do arquivo MP3
mapa_audio = {
    arquivo.stem: arquivo
    for arquivo in arquivos_mp3
}

def hash_arquivo_audio(audio_id):
    caminho = mapa_audio[audio_id]

    sha256 = hashlib.sha256()

    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(8192), b""):
            sha256.update(bloco)

    return sha256.hexdigest()

print("Quantidade de áudios no mapa:", len(mapa_audio))

Quantidade de áudios no mapa: 2216


In [30]:
df_audio = df_sem_duplicados_exatos.copy()

df_audio["Audio_Hash"] = df_audio["Audio_Song"].apply(hash_arquivo_audio)

print("Registros analisados:", len(df_audio))
print("Arquivos de áudio únicos pelo conteúdo:",
      df_audio["Audio_Hash"].nunique())

Registros analisados: 1323
Arquivos de áudio únicos pelo conteúdo: 1321


In [31]:
audios_repetidos = df_audio[
    df_audio.duplicated("Audio_Hash", keep=False)
].sort_values("Audio_Hash")

print("Linhas envolvidas em áudios idênticos:",
      len(audios_repetidos))

print("Grupos de áudios idênticos:",
      audios_repetidos["Audio_Hash"].nunique())

display(
    audios_repetidos[
        ["Artist", "Title", "Audio_Song", "Lyric_Song", "AllMusic Id", "Quadrant"]
    ]
)

Linhas envolvidas em áudios idênticos: 4
Grupos de áudios idênticos: 2


,Artist,Title,Audio_Song,Lyric_Song,AllMusic Id,Quadrant
165,Godsmack,I F****** Hate You,MT0003503243,MT0003503243,MT0003503243,Q2
1058,Godsmack,I Fucking Hate You,A028,L078,MT0044569810,Q2
220,Luther Vandross,Never Too Much,MT0011933875,MT0011933875,MT0011933875,Q1
931,Luther Vandross,Never Too Much - Remastered,A114-120,L004-120,MT0001671601,Q1


#### Comparação dos áudios

Após a remoção dos duplicados exatos, foram analisados 1.323 registros. A comparação do conteúdo dos arquivos MP3 através de SHA-256 identificou 1.321 conteúdos de áudio distintos.

Foram encontrados 4 registros, organizados em 2 grupos, cujos arquivos de áudio são exatamente idênticos:

- Godsmack — "I F****** Hate You" e "I Fucking Hate You";
- Luther Vandross — "Never Too Much" e "Never Too Much - Remastered".

Em ambos os casos, os identificadores dos registros são diferentes, mas os arquivos de áudio possuem exatamente o mesmo conteúdo. Isso demonstra que os identificadores e os títulos, isoladamente, não são suficientes para determinar se duas linhas correspondem à mesma gravação.

Entre as evidências analisadas, o conteúdo do áudio é a evidência mais forte para estabelecer que duas linhas representam a mesma gravação no dataset. Um mesmo título ou uma mesma letra podem corresponder a gravações diferentes, enquanto arquivos de áudio idênticos representam o mesmo conteúdo sonoro fornecido pelo dataset.

In [32]:
# Adicionar o hash das letras ao dataframe de áudio
df_comparacao = df_audio.copy()

df_comparacao["Lyric_Hash"] = df_comparacao["Lyric_Song"].apply(hash_letra)

# Selecionar apenas os registros com áudio idêntico
comparacao_audio = df_comparacao[
    df_comparacao.duplicated("Audio_Hash", keep=False)
].sort_values("Audio_Hash")

display(
    comparacao_audio[
        [
            "Artist",
            "Title",
            "Audio_Song",
            "Lyric_Song",
            "AllMusic Id",
            "Quadrant",
            "Audio_Hash",
            "Lyric_Hash"
        ]
    ]
)

,Artist,Title,Audio_Song,Lyric_Song,AllMusic Id,Quadrant,Audio_Hash,Lyric_Hash
165,Godsmack,I F****** Hate You,MT0003503243,MT0003503243,MT0003503243,Q2,68d4b4751b555b02873884bac0c2262f435ffd9be1aae8...,51ff2515aabd99eafe0546194d46cedc2ed6b0d38df131...
1058,Godsmack,I Fucking Hate You,A028,L078,MT0044569810,Q2,68d4b4751b555b02873884bac0c2262f435ffd9be1aae8...,51ff2515aabd99eafe0546194d46cedc2ed6b0d38df131...
220,Luther Vandross,Never Too Much,MT0011933875,MT0011933875,MT0011933875,Q1,a544b8f5100fa75ddc08b58c7b9edb0245ec27d856ea64...,0c049136c6fd7da3f579cb0a431d88cb9364aedda87940...
931,Luther Vandross,Never Too Much - Remastered,A114-120,L004-120,MT0001671601,Q1,a544b8f5100fa75ddc08b58c7b9edb0245ec27d856ea64...,1925d0051734615d30ebcd971769300f77b2d7cbf4ddcf...


In [33]:
for audio_hash, grupo in comparacao_audio.groupby("Audio_Hash"):

    print("=" * 70)

    print("Artistas:")
    print(grupo["Artist"].tolist())

    print("\nTítulos:")
    print(grupo["Title"].tolist())

    print("\nAudio_Song:")
    print(grupo["Audio_Song"].tolist())

    print("\nLyric_Song:")
    print(grupo["Lyric_Song"].tolist())

    print("\nAllMusic Id:")
    print(grupo["AllMusic Id"].tolist())

    print("\nMesmo conteúdo de letra:",
          grupo["Lyric_Hash"].nunique() == 1)

    print("Mesmo conteúdo de áudio:",
          grupo["Audio_Hash"].nunique() == 1)

    print()

Artistas:
['Godsmack', 'Godsmack']

Títulos:
['I F****** Hate You', 'I Fucking Hate You']

Audio_Song:
['MT0003503243', 'A028']

Lyric_Song:
['MT0003503243', 'L078']

AllMusic Id:
['MT0003503243', 'MT0044569810']

Mesmo conteúdo de letra: True
Mesmo conteúdo de áudio: True

Artistas:
['Luther Vandross', 'Luther  Vandross']

Títulos:
['Never Too Much', 'Never Too Much - Remastered']

Audio_Song:
['MT0011933875', 'A114-120']

Lyric_Song:
['MT0011933875', 'L004-120']

AllMusic Id:
['MT0011933875', 'MT0001671601']

Mesmo conteúdo de letra: False
Mesmo conteúdo de áudio: True



### 4.1.2 Comparação das evidências de duplicação

Foram comparados quatro tipos de evidência para determinar se duas linhas representam a mesma gravação: os identificadores, o artista e o título, o conteúdo das letras e o conteúdo do áudio.

As evidências não concordam em todos os casos. Após a remoção dos duplicados exatos, os identificadores `Audio_Song` e `Lyric_Song` não apresentaram repetições. O campo `AllMusic Id` apresentou repetições adicionais apenas devido ao valor `not_found`, que representa a ausência de um identificador válido e, portanto, não pode ser utilizado como evidência de duplicação.

A comparação das letras encontrou 18 registros organizados em 9 grupos com conteúdo textual idêntico. Entretanto, foram encontrados casos de letras iguais pertencentes a artistas diferentes, como "Twist and Shout" de The Isley Brothers e The Beatles. Portanto, uma letra idêntica pode indicar a mesma música ou composição, mas não necessariamente a mesma gravação.

A comparação dos arquivos de áudio encontrou 4 registros organizados em 2 grupos com conteúdo de áudio exatamente idêntico. No caso de Godsmack, os dois registros possuem identificadores e títulos diferentes, mas tanto a letra quanto o áudio são idênticos. No caso de Luther Vandross, os identificadores e os títulos são diferentes e as letras não são idênticas, mas os arquivos de áudio são exatamente iguais.

Assim, as diferentes fontes de evidência não concordam necessariamente. Entre as evidências analisadas, a igualdade do conteúdo do áudio é a evidência mais forte de que duas linhas representam a mesma gravação no dataset. Os identificadores, artista e título e as letras são úteis para encontrar candidatos, mas isoladamente não estabelecem que duas linhas correspondam à mesma gravação.

In [34]:
padrao_versoes = r"remix|remaster|live|edit|version|mix|acoustic"

versoes = df_sem_duplicados_exatos[
    df_sem_duplicados_exatos["Title"].str.contains(
        padrao_versoes,
        case=False,
        na=False,
        regex=True
    )
]

print("Registros com indicação de versão no título:", len(versoes))

display(
    versoes[
        ["Artist", "Title", "Audio_Song", "Lyric_Song", "Quadrant"]
    ].sort_values(["Artist", "Title"])
)

Registros com indicação de versão no título: 48


,Artist,Title,Audio_Song,Lyric_Song,Quadrant
495,Ace Hood,Mine - Remastered,MT0033717316,MT0033717316,Q1
621,Al Green,I Wanna Hold You (Remastered),MT0001829666,MT0001829666,Q1
1145,Alesana,"Congratulations, I Hate You - Remastered",A080-92,L130-92,Q2
1165,Barry White,"You're the First, the Last, My Everything - Re...",A160-149,L050-149,Q1
991,Bing Crosby,"Brother, Can You Spare a Dime? - Remastered",MT0004842694,MT0004842694,Q3
1338,Bing Crosby,Here Comes Santa Claus (Remastered),MT0000128573,MT0000128573,Q1
355,Blue,"Signed, Sealed, Delivered (I'm Yours)",MT0004425744,MT0004425744,Q1
1327,Bruce Springsteen,The Rising - Remastered,MT0006514225,MT0006514225,Q1
575,Christy Moore,Only Our Rivers Run Free [Live],MT0008428932,MT0008428932,Q3
416,Danity Kane,Heartbreaker - Remastered,MT0034064577,MT0034064577,Q1


In [35]:
padrao_versoes = r"\b(remix|remastered|remaster|live|edit|version|mix|acoustic)\b"

versoes = df_sem_duplicados_exatos[
    df_sem_duplicados_exatos["Title"].str.contains(
        padrao_versoes,
        case=False,
        na=False,
        regex=True
    )
]

print("Registros com indicação explícita de versão no título:", len(versoes))

display(
    versoes[
        ["Artist", "Title", "Audio_Song", "Lyric_Song", "Quadrant"]
    ].sort_values(["Artist", "Title"])
)

Registros com indicação explícita de versão no título: 43


C:\Users\Alicia\AppData\Local\Temp\ipykernel_20772\759212823.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_sem_duplicados_exatos["Title"].str.contains(


,Artist,Title,Audio_Song,Lyric_Song,Quadrant
495,Ace Hood,Mine - Remastered,MT0033717316,MT0033717316,Q1
621,Al Green,I Wanna Hold You (Remastered),MT0001829666,MT0001829666,Q1
1145,Alesana,"Congratulations, I Hate You - Remastered",A080-92,L130-92,Q2
1165,Barry White,"You're the First, the Last, My Everything - Re...",A160-149,L050-149,Q1
991,Bing Crosby,"Brother, Can You Spare a Dime? - Remastered",MT0004842694,MT0004842694,Q3
1338,Bing Crosby,Here Comes Santa Claus (Remastered),MT0000128573,MT0000128573,Q1
1327,Bruce Springsteen,The Rising - Remastered,MT0006514225,MT0006514225,Q1
575,Christy Moore,Only Our Rivers Run Free [Live],MT0008428932,MT0008428932,Q3
416,Danity Kane,Heartbreaker - Remastered,MT0034064577,MT0034064577,Q1
1148,Das EFX,Knockin' Niggaz Off (Remastered),MT0002460914,MT0002460914,Q2


In [36]:
padrao_versoes_explicitas = (
    r"\[[^\]]*(?:live|remix|version|edit|acoustic)[^\]]*\]"
    r"|\([^)]*(?:live|remix|remastered|remaster|version|edit|acoustic)[^)]*\)"
    r"|-\s*(?:remastered|remaster|live|remix|edit|acoustic)\b"
)

versoes_explicitas = df_sem_duplicados_exatos[
    df_sem_duplicados_exatos["Title"].str.contains(
        padrao_versoes_explicitas,
        case=False,
        na=False,
        regex=True
    )
]

print("Registros com indicação explícita de versão:",
      len(versoes_explicitas))

display(
    versoes_explicitas[
        ["Artist", "Title", "Audio_Song", "Lyric_Song", "Quadrant"]
    ].sort_values(["Artist", "Title"])
)

Registros com indicação explícita de versão: 41


,Artist,Title,Audio_Song,Lyric_Song,Quadrant
495,Ace Hood,Mine - Remastered,MT0033717316,MT0033717316,Q1
621,Al Green,I Wanna Hold You (Remastered),MT0001829666,MT0001829666,Q1
1145,Alesana,"Congratulations, I Hate You - Remastered",A080-92,L130-92,Q2
1165,Barry White,"You're the First, the Last, My Everything - Re...",A160-149,L050-149,Q1
991,Bing Crosby,"Brother, Can You Spare a Dime? - Remastered",MT0004842694,MT0004842694,Q3
1338,Bing Crosby,Here Comes Santa Claus (Remastered),MT0000128573,MT0000128573,Q1
1327,Bruce Springsteen,The Rising - Remastered,MT0006514225,MT0006514225,Q1
575,Christy Moore,Only Our Rivers Run Free [Live],MT0008428932,MT0008428932,Q3
416,Danity Kane,Heartbreaker - Remastered,MT0034064577,MT0034064577,Q1
1148,Das EFX,Knockin' Niggaz Off (Remastered),MT0002460914,MT0002460914,Q2


In [37]:
import re
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).lower().strip()

    # Remover acentos
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )

    # Remover informações explícitas de versão entre [] ou ()
    texto = re.sub(
        r"\[[^\]]*(?:live|remix|remastered|remaster|version|edit|acoustic)[^\]]*\]",
        "",
        texto
    )

    texto = re.sub(
        r"\([^)]*(?:live|remix|remastered|remaster|version|edit|acoustic)[^)]*\)",
        "",
        texto
    )

    # Remover indicações após hífen
    texto = re.sub(
        r"\s*-\s*(?:remastered|remaster|live|remix|edit|acoustic).*$",
        "",
        texto
    )

    # Remover espaços extras
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


df_versoes = df_sem_duplicados_exatos.copy()

df_versoes["Artist_Normalizado"] = (
    df_versoes["Artist"].apply(normalizar_texto)
)

df_versoes["Title_Base"] = (
    df_versoes["Title"].apply(normalizar_texto)
)

print(
    df_versoes[
        ["Artist", "Title", "Artist_Normalizado", "Title_Base"]
    ].head(20)
)

              Artist                        Title Artist_Normalizado  \
0   Marlene Dietrich        Falling in Love Again   marlene dietrich   
1      George Strait  Four Down and Twelve Across      george strait   
2   Gorilla Biscuits                    Forgotten   gorilla biscuits   
3      Nat King Cole      Baby, Baby All the Time      nat king cole   
4     Merle Haggard        I Made the Prison Band      merle haggard   
5      Travis  Tritt              Help Me Hold On       travis tritt   
6    Ella Fitzgerald                  Cryin' Mood    ella fitzgerald   
7     Billie Holiday               My Last Affair     billie holiday   
8    Revolting Cocks                    Mr. Lucky    revolting cocks   
9        johnny cash             Old Apache Squaw        johnny cash   
10       The Beatles          When I'm Sixty-Four        the beatles   
11       Sheek Louch                     Pressure        sheek louch   
12    Paint It Black                  We Will Not     paint it b

In [38]:
# Procurar músicas que possuem o mesmo artista e o mesmo título-base

grupos_mesma_musica = df_versoes[
    df_versoes.duplicated(
        subset=["Artist_Normalizado", "Title_Base"],
        keep=False
    )
].sort_values(
    ["Artist_Normalizado", "Title_Base"]
)

print("Linhas envolvidas:", len(grupos_mesma_musica))

print(
    "Grupos de possíveis versões da mesma música:",
    grupos_mesma_musica[
        ["Artist_Normalizado", "Title_Base"]
    ].drop_duplicates().shape[0]
)

display(
    grupos_mesma_musica[
        [
            "Artist",
            "Title",
            "Audio_Song",
            "Lyric_Song",
            "AllMusic Id",
            "Quadrant",
            "Artist_Normalizado",
            "Title_Base"
        ]
    ]
)

Linhas envolvidas: 14
Grupos de possíveis versões da mesma música: 7


,Artist,Title,Audio_Song,Lyric_Song,AllMusic Id,Quadrant,Artist_Normalizado,Title_Base
480,Gloria Estefan,Rhythm Is Gonna Get You,MT0003965796,MT0003965796,MT0003965796,Q1,gloria estefan,rhythm is gonna get you
686,Gloria Estefan,Rhythm Is Gonna Get You,MT0007132421,MT0007132421,MT0007132421,Q1,gloria estefan,rhythm is gonna get you
220,Luther Vandross,Never Too Much,MT0011933875,MT0011933875,MT0011933875,Q1,luther vandross,never too much
931,Luther Vandross,Never Too Much - Remastered,A114-120,L004-120,MT0001671601,Q1,luther vandross,never too much
149,Madonna,Deeper and Deeper,MT0004692456,MT0004692456,MT0004692456,Q1,madonna,deeper and deeper
926,Madonna,Deeper and Deeper [Album Edit],MT0002294387,MT0002294387,MT0002294387,Q1,madonna,deeper and deeper
268,Madonna,Like a Virgin [Extended Dance Remix],MT0001284282,MT0001284282,MT0001284282,Q1,madonna,like a virgin
477,Madonna,Like a Virgin,MT0028241237,MT0028241237,MT0028241237,Q1,madonna,like a virgin
203,Marvin Gaye,Stubborn Kind of Fellow [Single Version (Mono)],MT0012521149,MT0012521149,MT0012521149,Q1,marvin gaye,stubborn kind of fellow
850,Marvin Gaye,Stubborn Kind of Fellow,MT0011957429,MT0011957429,MT0011957429,Q1,marvin gaye,stubborn kind of fellow


In [39]:
# Adicionar os hashes de letra e áudio à tabela de versões
df_versoes["Lyric_Hash"] = df_letras["Lyric_Hash"]
df_versoes["Audio_Hash"] = df_audio["Audio_Hash"]

# Selecionar apenas os grupos que representam possíveis versões da mesma música
candidatos = df_versoes[
    df_versoes.duplicated(
        subset=["Artist_Normalizado", "Title_Base"],
        keep=False
    )
].copy()

# Resumir as evidências de cada grupo
resumo_candidatos = (
    candidatos
    .groupby(["Artist_Normalizado", "Title_Base"])
    .agg(
        Registros=("Title", "size"),
        Titulos=("Title", lambda x: list(x)),
        Audio_IDs=("Audio_Song", lambda x: list(x)),
        Lyric_IDs=("Lyric_Song", lambda x: list(x)),
        AllMusic_IDs=("AllMusic Id", lambda x: list(x)),
        Audios_Diferentes=("Audio_Hash", "nunique"),
        Letras_Diferentes=("Lyric_Hash", "nunique")
    )
    .reset_index()
)

display(resumo_candidatos)

,Artist_Normalizado,Title_Base,Registros,Titulos,Audio_IDs,Lyric_IDs,AllMusic_IDs,Audios_Diferentes,Letras_Diferentes
0,gloria estefan,rhythm is gonna get you,2,"[Rhythm Is Gonna Get You, Rhythm Is Gonna Get ...","[MT0003965796, MT0007132421]","[MT0003965796, MT0007132421]","[MT0003965796, MT0007132421]",2,2
1,luther vandross,never too much,2,"[Never Too Much, Never Too Much - Remastered]","[MT0011933875, A114-120]","[MT0011933875, L004-120]","[MT0011933875, MT0001671601]",1,2
2,madonna,deeper and deeper,2,"[Deeper and Deeper, Deeper and Deeper [Album E...","[MT0004692456, MT0002294387]","[MT0004692456, MT0002294387]","[MT0004692456, MT0002294387]",2,2
3,madonna,like a virgin,2,"[Like a Virgin [Extended Dance Remix], Like a ...","[MT0001284282, MT0028241237]","[MT0001284282, MT0028241237]","[MT0001284282, MT0028241237]",2,2
4,marvin gaye,stubborn kind of fellow,2,[Stubborn Kind of Fellow [Single Version (Mono...,"[MT0012521149, MT0011957429]","[MT0012521149, MT0011957429]","[MT0012521149, MT0011957429]",2,2
5,prince,kiss,2,"[Kiss - Remastered, kiss]","[MT0002343207, A105-117]","[MT0002343207, L155-117]","[MT0002343207, MT0008266225]",2,1
6,prince,u got the look,2,"[U Got the Look, U GOT THE LOOK]","[MT0004444570, A005]","[MT0004444570, L055]","[MT0004444570, MT0005204984]",2,1


In [40]:
display(
    resumo_candidatos[
        [
            "Artist_Normalizado",
            "Title_Base",
            "Registros",
            "Audios_Diferentes",
            "Letras_Diferentes"
        ]
    ]
)

,Artist_Normalizado,Title_Base,Registros,Audios_Diferentes,Letras_Diferentes
0,gloria estefan,rhythm is gonna get you,2,2,2
1,luther vandross,never too much,2,1,2
2,madonna,deeper and deeper,2,2,2
3,madonna,like a virgin,2,2,2
4,marvin gaye,stubborn kind of fellow,2,2,2
5,prince,kiss,2,2,1
6,prince,u got the look,2,2,1


### 4.1.2 Comparação das evidências de duplicação

Foram comparados quatro tipos de evidência: os identificadores, a combinação artista/título, o conteúdo das letras e o conteúdo dos arquivos de áudio.

Os identificadores são úteis para encontrar possíveis duplicados, mas não são suficientes por si só. Após a remoção das duplicações exatas, `Audio_Song` e `Lyric_Song` não apresentaram identificadores repetidos, enquanto as repetições adicionais de `AllMusic Id` correspondiam ao valor `not_found`, que representa a ausência de um identificador válido.

A combinação entre artista e título também não é suficiente para estabelecer que duas linhas correspondem à mesma gravação, pois a mesma música pode possuir diferentes versões, como versões remasterizadas, ao vivo ou remixes.

A comparação das letras mostrou que duas gravações diferentes podem possuir a mesma letra. Por exemplo, `Kiss` e `Kiss - Remastered`, de Prince, apresentam o mesmo conteúdo de letra, mas conteúdos de áudio diferentes. Portanto, a igualdade das letras identifica fortemente a mesma composição, mas não necessariamente a mesma gravação.

A evidência mais forte para estabelecer que duas linhas correspondem à mesma gravação é o conteúdo do áudio. A comparação por hash permite identificar arquivos de áudio exatamente iguais independentemente dos identificadores ou metadados utilizados. O conjunto de dados contém, por exemplo, duas entradas de `Never Too Much`, de Luther Vandross, com identificadores e letras diferentes, mas com o mesmo conteúdo de áudio.

Assim, artista e título ajudam a identificar a mesma música, as letras fornecem evidência sobre a composição, enquanto a igualdade do conteúdo de áudio estabelece de forma mais direta que duas entradas contêm a mesma gravação.

### 4.1.3 Remixes, edits e versões ao vivo

Um remix, edit, versão ao vivo ou outra versão alternativa não deve ser tratado como uma duplicação da gravação original. Embora possa representar a mesma música ou composição, trata-se de uma gravação diferente.

Os dados confirmam essa distinção. Após normalizar os títulos para agrupar diferentes versões da mesma música, foram encontrados casos em que duas entradas pertencem à mesma música, mas possuem conteúdos de áudio diferentes. Isso ocorre, por exemplo, em grupos como `Kiss`, de Prince.

Portanto, essas versões podem ser consideradas duplicadas ao nível da música/composição, mas não ao nível da gravação. Elas devem ser mantidas como gravações distintas.

In [41]:
# 4.1.4 - Carregar a partição de teste

df_test = pd.read_csv("test_songs.csv")

print("Número de linhas:", len(df_test))
print("\nColunas:")
print(df_test.columns.tolist())

display(df_test.head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'test_songs.csv'

In [42]:
from pathlib import Path

print("Pasta atual:")
print(Path.cwd())

print("\nArquivos CSV encontrados:")
for arquivo in Path(".").rglob("*.csv"):
    print(arquivo)

Pasta atual:
C:\Users\Alicia\Downloads\dataset\build\MERGE_ECAC

Arquivos CSV encontrados:
merge_ecac\av_values.csv
merge_ecac\metadata.csv
merge_ecac\test_songs.csv
merge_original\av_values.csv
merge_original\metadata.csv
truth\truth_cells.csv
truth\truth_rows.csv


In [43]:
df_test = pd.read_csv("merge_ecac/test_songs.csv")

print("Número de linhas:", len(df_test))

print("\nColunas:")
print(df_test.columns.tolist())

display(df_test.head(10))

Número de linhas: 409

Colunas:
['Audio_Song', 'Quadrant']


,Audio_Song,Quadrant
0,MT0009263768,Q3
1,MT0030214520,Q2
2,MT0009005900,Q1
3,MT0000092637,Q3
4,MT0000638634,Q4
5,MT0004400843,Q3
6,MT0008953247,Q3
7,MT0014953592,Q2
8,MT0028698324,Q3
9,A153-130,Q1


In [44]:
test_ids = set(df_test["Audio_Song"])

metadata_ids = set(df_metadata["Audio_Song"])

presentes = test_ids.intersection(metadata_ids)
ausentes = test_ids.difference(metadata_ids)

print("Músicas no conjunto de teste:", len(test_ids))
print("Encontradas no metadata:", len(presentes))
print("Não encontradas no metadata:", len(ausentes))

if len(ausentes) > 0:
    print("\nIDs não encontrados:")
    print(sorted(ausentes))

Músicas no conjunto de teste: 409
Encontradas no metadata: 409
Não encontradas no metadata: 0


In [45]:
duplicados_exatos = df_metadata[
    df_metadata.duplicated(keep=False)
].copy()

duplicados_exatos["Esta_no_Teste"] = (
    duplicados_exatos["Audio_Song"].isin(test_ids)
)

print("Linhas envolvidas em duplicação exata:", len(duplicados_exatos))
print(
    "Linhas duplicadas que pertencem ao teste:",
    duplicados_exatos["Esta_no_Teste"].sum()
)

display(
    duplicados_exatos[
        ["Audio_Song", "Artist", "Title", "Quadrant", "Esta_no_Teste"]
    ].sort_values(["Audio_Song"])
)

Linhas envolvidas em duplicação exata: 50
Linhas duplicadas que pertencem ao teste: 14


,Audio_Song,Artist,Title,Quadrant,Esta_no_Teste
971,A071-70,Ben Folds Five,Brick,Q3,False
131,A071-70,Ben Folds Five,Brick,Q3,False
103,A132-121,The Ten Tenors,"Bee Gees Medley: Staying Alive, You Should Be ...",Q1,False
556,A132-121,The Ten Tenors,"Bee Gees Medley: Staying Alive, You Should Be ...",Q1,False
1108,MT0000408441,Lightnin' Hopkins,Give Me Back That Wig (Wig Wearing Woman),Q3,False
338,MT0000408441,Lightnin' Hopkins,Give Me Back That Wig (Wig Wearing Woman),Q3,False
544,MT0000685538,Carl Perkins,Let the Jukebox Keep on Playing,Q3,False
640,MT0000685538,Carl Perkins,Let the Jukebox Keep on Playing,Q3,False
179,MT0000886390,Dolly Parton,Randy,Q1,False
378,MT0000886390,Dolly Parton,Randy,Q1,False


In [46]:
duplicados_teste = duplicados_exatos[
    duplicados_exatos["Esta_no_Teste"]
]

print("Linhas duplicadas pertencentes ao teste:", len(duplicados_teste))
print(
    "Músicas distintas do teste envolvidas em duplicação:",
    duplicados_teste["Audio_Song"].nunique()
)

display(
    duplicados_teste[
        ["Audio_Song", "Artist", "Title", "Quadrant"]
    ].sort_values("Audio_Song")
)

Linhas duplicadas pertencentes ao teste: 14
Músicas distintas do teste envolvidas em duplicação: 7


,Audio_Song,Artist,Title,Quadrant
353,MT0001189913,Kay Kyser,"(I Got Spurs That) Jingle, Jangle, Jingle",Q1
701,MT0001189913,Kay Kyser,"(I Got Spurs That) Jingle, Jangle, Jingle",Q1
154,MT0004489039,bob dylan,North Country Blues,Q3
848,MT0004489039,bob dylan,North Country Blues,Q3
325,MT0005155949,Carmen McRae,How I Wish (Ask Me Now),Q3
777,MT0005155949,Carmen McRae,How I Wish (Ask Me Now),Q3
349,MT0007644103,Al Green,I Can't Stop,Q1
964,MT0007644103,Al Green,I Can't Stop,Q1
929,MT0009360971,Megadeth,Black Curtains,Q2
1235,MT0009360971,Megadeth,Black Curtains,Q2


In [47]:
# Identificar quais registros pertencem ao conjunto de teste
df_audio["Esta_no_Teste"] = df_audio["Audio_Song"].isin(test_ids)

# Para cada conteúdo de áudio, verificar se aparece
# tanto no treino quanto no teste
hash_treino_teste = (
    df_audio.groupby("Audio_Hash")["Esta_no_Teste"]
    .nunique()
)

hashes_vazamento = hash_treino_teste[
    hash_treino_teste > 1
].index

possivel_vazamento = df_audio[
    df_audio["Audio_Hash"].isin(hashes_vazamento)
].sort_values("Audio_Hash")

print(
    "Áudios idênticos presentes tanto no treino quanto no teste:",
    len(hashes_vazamento)
)

print(
    "Linhas envolvidas:",
    len(possivel_vazamento)
)

display(
    possivel_vazamento[
        [
            "Artist",
            "Title",
            "Audio_Song",
            "Lyric_Song",
            "Quadrant",
            "Esta_no_Teste"
        ]
    ]
)

Áudios idênticos presentes tanto no treino quanto no teste: 1
Linhas envolvidas: 2


,Artist,Title,Audio_Song,Lyric_Song,Quadrant,Esta_no_Teste
165,Godsmack,I F****** Hate You,MT0003503243,MT0003503243,Q2,False
1058,Godsmack,I Fucking Hate You,A028,L078,Q2,True


In [48]:
# Remover primeiro as 25 cópias exatas do metadata
df_limpo = df_metadata.drop_duplicates().copy()

# Separar os registros usando a partição oficial de teste
df_teste = df_limpo[
    df_limpo["Audio_Song"].isin(test_ids)
].copy()

df_treino = df_limpo[
    ~df_limpo["Audio_Song"].isin(test_ids)
].copy()

print("Metadata original:", len(df_metadata))
print("Após remover duplicados exatos:", len(df_limpo))
print("Treino:", len(df_treino))
print("Teste:", len(df_teste))
print("Total treino + teste:", len(df_treino) + len(df_teste))

Metadata original: 1348
Após remover duplicados exatos: 1323
Treino: 914
Teste: 409
Total treino + teste: 1323


In [49]:
# Remover do treino a gravação que possui áudio idêntico
# a uma música presente no conjunto oficial de teste

id_vazamento_treino = "MT0003503243"

print("Antes da remoção:", len(df_treino))

df_treino = df_treino[
    df_treino["Audio_Song"] != id_vazamento_treino
].copy()

print("Depois da remoção:", len(df_treino))

print(
    "Registro ainda presente no treino:",
    (df_treino["Audio_Song"] == id_vazamento_treino).sum()
)

Antes da remoção: 914
Depois da remoção: 913
Registro ainda presente no treino: 0


In [50]:
# Considerar apenas os registros que permaneceram após a limpeza
ids_treino_limpo = set(df_treino["Audio_Song"])
ids_teste_limpo = set(df_teste["Audio_Song"])

df_audio_final = df_audio[
    df_audio["Audio_Song"].isin(ids_treino_limpo | ids_teste_limpo)
].copy()

df_audio_final["Conjunto"] = df_audio_final["Audio_Song"].apply(
    lambda x: "Teste" if x in ids_teste_limpo else "Treino"
)

# Verificar se algum mesmo conteúdo de áudio aparece nos dois conjuntos
verificacao = (
    df_audio_final.groupby("Audio_Hash")["Conjunto"]
    .nunique()
)

hashes_entre_conjuntos = verificacao[
    verificacao > 1
].index

print(
    "Áudios idênticos presentes simultaneamente em treino e teste:",
    len(hashes_entre_conjuntos)
)

print("Registros finais de treino:", len(df_treino))
print("Registros finais de teste:", len(df_teste))
print("Total final:", len(df_treino) + len(df_teste))

Áudios idênticos presentes simultaneamente em treino e teste: 0
Registros finais de treino: 913
Registros finais de teste: 409
Total final: 1322


In [51]:
# 4.2 Valores ausentes

# Juntar treino e teste apenas para analisar a qualidade dos metadados
df_final = pd.concat([df_treino, df_teste], ignore_index=True)

valores_ausentes = df_final.isna().sum()

valores_ausentes = valores_ausentes[
    valores_ausentes > 0
].sort_values(ascending=False)

print("Total de registros analisados:", len(df_final))
print("Colunas com valores ausentes:", len(valores_ausentes))

display(
    pd.DataFrame({
        "Valores_Ausentes": valores_ausentes,
        "Percentual": (valores_ausentes / len(df_final) * 100).round(2)
    })
)

Total de registros analisados: 1322
Colunas com valores ausentes: 20


,Valores_Ausentes,Percentual
num_Genres,1312,99.24
num_MoodsAll,1312,99.24
LowestYear,498,37.67
ActualYear,306,23.15
Duration,271,20.50
Relevance,244,18.46
Year,194,14.67
AppearancesTrackIDs,186,14.07
AppearancesAlbumIDs,186,14.07
ThemeWeights,185,13.99


In [52]:
# 4.2.1 Verificar valores ausentes nas colunas essenciais

colunas_essenciais = [
    "Audio_Song",
    "Lyric_Song",
    "Quadrant",
    "Artist",
    "Title"
]

ausentes_essenciais = df_final[colunas_essenciais].isna().sum()

print("Valores ausentes nas colunas essenciais:")
display(ausentes_essenciais.to_frame(name="Valores_Ausentes"))

print(
    "\nTotal de linhas com pelo menos um valor essencial ausente:",
    df_final[colunas_essenciais].isna().any(axis=1).sum()
)

Valores ausentes nas colunas essenciais:


,Valores_Ausentes
Audio_Song,0
Lyric_Song,0
Quadrant,0
Artist,0
Title,0



Total de linhas com pelo menos um valor essencial ausente: 0


In [53]:
# 4.2.2 - Verificar os quadrantes após a limpeza

print("Quadrantes no treino:")
print(df_treino["Quadrant"].value_counts().sort_index())

print("\nQuadrantes no teste:")
print(df_teste["Quadrant"].value_counts().sort_index())

print("\nValores inesperados no treino:")
print(
    df_treino.loc[
        ~df_treino["Quadrant"].isin(["Q1", "Q2", "Q3", "Q4"]),
        "Quadrant"
    ].unique()
)

print("\nValores inesperados no teste:")
print(
    df_teste.loc[
        ~df_teste["Quadrant"].isin(["Q1", "Q2", "Q3", "Q4"]),
        "Quadrant"
    ].unique()
)

Quadrantes no treino:
Quadrant
Q1    366
Q2    111
Q3    347
Q4     89
Name: count, dtype: int64

Quadrantes no teste:
Quadrant
Q1    159
Q2     56
Q3    153
Q4     41
Name: count, dtype: int64

Valores inesperados no treino:
<StringArray>
[]
Length: 0, dtype: str

Valores inesperados no teste:
<StringArray>
[]
Length: 0, dtype: str


In [54]:
# 4.2.3 - Guardar os conjuntos limpos

from pathlib import Path

output_dir = Path("cleaned")
output_dir.mkdir(exist_ok=True)

treino_path = output_dir / "train_clean.csv"
teste_path = output_dir / "test_clean.csv"

df_treino.to_csv(treino_path, index=False)
df_teste.to_csv(teste_path, index=False)

print("Treino guardado em:", treino_path)
print("Teste guardado em:", teste_path)

print("\nDimensões finais:")
print("Treino:", df_treino.shape)
print("Teste:", df_teste.shape)

Treino guardado em: cleaned\train_clean.csv
Teste guardado em: cleaned\test_clean.csv

Dimensões finais:
Treino: (913, 27)
Teste: (409, 27)


## 5. Resultado final da limpeza

Após a análise e limpeza dos dados, foram obtidos:

- **913 registos de treino**
- **409 registos de teste**
- **1322 registos no total**
- Nenhum valor ausente nas colunas essenciais
- Nenhum valor inválido na variável `Quadrant`
- Duplicações exatas removidas
- Sobreposição de conteúdo de áudio entre treino e teste removida

Os conjuntos finais foram guardados em:

- `cleaned/train_clean.csv`
- `cleaned/test_clean.csv`

Estes ficheiros serão utilizados nas etapas seguintes de extração de características e modelação.